# Meshtastic Signal Analysis

This notebook looks at LoRa-like chirp captures. It accepts a local Meshtastic capture if present, but otherwise synthesizes a packet-like burst so the spectrogram and timing measurements remain available.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


In [ ]:
capture_path = CAPTURE_ROOT / "meshtastic_iq.npz"
if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 500_000
    chirps = []
    for sf in [7, 7, 9, 10]:
        symbol_time = (2 ** sf) / 125_000
        t = np.arange(0, symbol_time, 1 / fs_iq)
        chirp = signal.chirp(t, f0=-62_500, f1=62_500, t1=t[-1], method="linear")
        chirps.append(np.exp(1j * np.angle(signal.hilbert(chirp))))
        chirps.append(np.zeros(int(0.02 * fs_iq), dtype=np.complex128))
    iq = np.concatenate(chirps)
    print("Using synthetic LoRa-like chirp burst fallback.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_waveform(np.real(iq[:20000]), fs=fs_iq, ax=axes[0], title="IQ real part")
plot_spectrogram(np.real(iq), fs=fs_iq, ax=axes[1], title="Chirp spectrogram", nperseg=1024, noverlap=768)
axes[1].set_ylim(0, 150_000)
plt.tight_layout()

ener = np.abs(iq)
active = ener > (0.2 * np.max(ener))
burst_duration_ms = np.sum(active) / fs_iq * 1e3
display(Markdown(f"**Approximate active burst time:** {burst_duration_ms:.2f} ms"))


## Key Takeaway

Meshtastic-style signals are easiest to reason about in the time-frequency plane. The chirp structure is the feature, not an implementation detail.